1. **Nome(s):** Leonardo Silveira Kiepper

2. **Matrícula(s):** 2026140417

3. **E-mail(s) institucional(is):** leonardo.kiepper@edu.ufes.br

3. **Curso(s):** Doutorado em Ciência da Computação

4. **Link do seu Colab com suas soluções (para que o professor possa acessar e rodar o código):** https://colab.research.google.com/github/LeoKiepper/Ciencia-de-Dados/blob/main/trabalho1/T1_leonardo_kiepper.ipynb


# Introdução

Cada compra que uma pessoa faz gera dados que são úteis para diversas aplicações práticas. Por exemplo, em se tratando de artigos de mercearia ou supermercado, ter um registro dos preços de produtos de interesse permite o monitoramento de gastos pessoais, precificação de receitas, etc. De modo geral, para tais aplicações serem viáveis em escala pessoal, é fundamental que a solução seja de baixo custo e de baixa manutenção, o que elimina soluções como controle de estoque informatizado, que muitos supermercados usam.

Por exigência de lei, quando requisitado, a maioria dos estabelecimentos já entrega o cupom fiscal detalhando a compra. Sendo assim, o cupom fiscal é o artefato físico natural de onde extrair os dados necessários para estas aplicaçãoes. No entanto, a tarefa de ler cupons fiscais e extrair esses dados manualmente não é prática, devido ao volume de dados necessário para que a solução implementada tenha utilidade.

De modo geral, os dados importantes para tais aplicações, que estariam presentes em uma nota ou cupom fiscal impresso, não são prontamente utilizáveis. Além da etapa de digitalização para converter em arquivo digital, e obrigatoriamente o reconhecimento óptico de caracteres (OCR) para converter imagem para texto, é necessário uma passada de extração de dados estruturada, na qual dados como descrição do produto e valor, são individualizados num arquivo tabular. Os dados mínimos de interesse são a descrição do produto, e o valor.

Mesmo tendo feito uma leitura e extração estruturada perfeitas, isto é, de modo que os dados constando no cupom fiscal físico tenham sido copiados exatamente para o arquivo digital, os dados ainda precisam ser curados, especificamente a descrição do produto. O desafio aqui é que a descrição do produto constando num cupom fiscal é extremamente truncada, para caber na dimensão limitada do papel térmico que normalmente é utilizado para esse propósito. Soma-se a isso os fatos de que cada comércio descreve e abrevia o produto de sua forma particular, e que, depois de identificado o produto a partir da descrição, é necessário adotar alguma forma padronizada de indexar o produto identificado para que a solução implementada seja amplamente aplicável a situações reais, e fica evidente que o problema não é trivial.

Limitando o escopo do projeto a predizer a descrição completa a partir da descrição truncada, procurou-se um conjunto de dados que suportasse essa tarefa. A base de dados aberta ELEVEN (https://github.com/vinidiol/descmerc) traz aproximadamente 1,1 milhão de linhas retiradas de nota fiscais anonimizadas, constando: a descrição do produto truncada, um código na Nomenclatura Comum do Mercosul (NCM) declarado pelo emissor, e um rótulo anotado por auditores fiscais especialistas. 

In [91]:
%pip install tensorflow
%pip install itables

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
#@title Funções auxiliares
import os, joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

DIR_ART = 'artefatos'

def _caminho(nome, ext):
    return os.path.join(DIR_ART, f'{nome}.{ext}')

def salva(nome, obj):
    """Matriz esparsa -> .npz; qualquer outro objeto -> .joblib."""
    os.makedirs(DIR_ART, exist_ok=True)
    if sp.issparse(obj):
        sp.save_npz(_caminho(nome, 'npz'), obj.tocsr())
    else:
        joblib.dump(obj, _caminho(nome, 'joblib'), compress=3)

def carrega(nome):
    if os.path.exists(_caminho(nome, 'npz')):
        return sp.load_npz(_caminho(nome, 'npz'))
    return joblib.load(_caminho(nome, 'joblib'))

def existe(nome):
    return any(os.path.exists(_caminho(nome, e)) for e in ('npz', 'joblib'))

def comoDataFrame(M, vocab, indice=None, linhas=slice(0, 20), so_nao_nulas=True):
    """Fatia de M como DataFrame, para visualização. `linhas`: slice ou lista de posições.
    Com so_nao_nulas=True, mostra só as colunas com algum valor nas linhas escolhidas."""
    sub = M[linhas]
    idx = None if indice is None else pd.Index(indice)[linhas]
    cols = np.unique(sub.indices) if so_nao_nulas else np.arange(M.shape[1])
    return pd.DataFrame(sub[:, cols].toarray(), index=idx, columns=np.asarray(vocab)[cols])


In [3]:
#@title Carrega o dataset

import os
import pandas as pd
from langdetect import detect, DetectorFactory, LangDetectException
from itables import show
# Carrega a partir do github
base = pd.read_csv('https://github.com/LeoKiepper/Ciencia-de-Dados/raw/main/trabalho2/descmerc/BaseDESC_NCM.zip',
                   sep='\t', dtype={'XPROD': str, 'NCM': str})

show(base)
show(base['NCM'].value_counts())
print("Quantas classes resultam em cada nível do NCM")
print({k: base['NCM'].str[:k].nunique() for k in (2, 4, 6, 8)})


Quantas classes resultam em cada nível do NCM
{2: 82, 4: 703, 6: 1628, 8: 2174}


# Metodogia

### Pré-processamento

Foi selecionado um algoritimo de classificação para exprimir relações entre a descrição do produto (coluna "XPROD" do *dataset*, entrada da rede) e o código NCM (saída, ou predição, da rede). A coluna "XPROD" tem dados formatados como *string*. Para tratar a imprevisibilidade do estilo de escrita de cada descrição, foi definida a função de preprocessamento abaixo.

A lógica por trás dessa função é remover ruído das descrições que não contribuiriam para descrever o produto, tomando cuidado para não remover sinal junto com o ruído, levando em conta características específicas de como pessoas escrevem essas descrições. Duas regras globais implementadas:

>"Qualificadores que sejam aplicáveis a múltiplos produtos de múltiplas categorias não contribuem para individualizar o produto para uma categoria, então são ruído". (Por exemplo, características da embalagem, marketing como 'leve X pague Y', quantidades)

>"Se há risco de eliminar sinal, não remover. A informação passa a ser dado para o algoritimo aprender ou ignorar."

Um caso notável de qualificadores que foram mantidos pela segunda regra são cores. Para alguns produtos, a cor é um qualificador que não muda a função. Por exemplo, uma escova de dente pode ser de qualquer cor, e ter exatamente a mesma função. Para outros, a cor é um qualificador que distingue o produto de outros semelhantes. Por exemplo, vinho branco é diferente de vinho tinto.

Com base nessas duas regras, foi criada uma sequência de regex para filtrar as strings. Esses regex removem grafias observadas para quantidades (que em geral não seguem grafia padronizada de unidades de medida, como 'metro' → 'm', 'litro' → 'l', 'grama' → 'g, etc). Depois de filtradas, as strings são tokenizadas. Alguns problemas são atacados em mais de uma etapa, para retirá-los de formas diferentes que não são fáceis de pegar por uma regra única. São feitas as seguintes operações. Para listagens completas das chaves regex usadas, vide o código abaixo.

1. Remover acentos e quantidades/tamanhos de embalagem
2. Dividir o texto em tokens
3. Converter o texto para letras minúsculas
4. Separar tokens que usam '.' ou '/' como separador
5. Remover símbolos de pontuação de cada token e descartar tokens vazios
6. Separar palavra colada a uma quantidade (fugini100gr -> fugini, 100gr)
7. Remover unidades, embalagens, marketing, tamanho, referência e stop words
8. Remover letras soltas (tokens com apenas uma letra)
9. Remover números soltos (3, 1/2, 030003, 20x25) e quantidades (750ml, 1kg)
10. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)


In [ ]:
#@title Tokeniza o dataset
import re, unicodedata
import pandas as pd
import nltk
import string
from nltk import word_tokenize
from nltk.corpus import stopwords
from itables import show
from collections import Counter
from functools import lru_cache

nltk.download('punkt_tab')
nltk.download('stopwords')

NUM  = r'\d+(?:[.,]\d+)?'
UNID = r'(?:M2|MLN|ML|MLS|M|MT|MTS|G|GR|GRS|KG|KGS|L|LT|LTS|LI|LITROS?|UN|UND|UNDS|UNID|U|PC|PCS|KIT)'
EMB  = r'(?:CX|CXA|UN|UND|PT|PC|FD|DP|SC|CJ|PCT|BD|FR|GA|GF|DZ)'
FIM  = r'\.?(?![A-Z\d])'          # fim do token: não seguido de letra ou dígito
PADRAO_QTD = re.compile(rf'''
    (?<![\w,.]){EMB}?(?:{NUM}\s?X\s?)+{NUM}\s?{UNID}{FIM}   # 12X500ML, DP12X24X22G
  | (?<![\w,.]){EMB}(?:{NUM}\s?X\s?)+{NUM}{FIM}             # CX20X50
  | (?<![\w,.])1\s?X\s?\d+{FIM}                             # 1 X 12
  | (?<![\w,.]){EMB}?{NUM}\s?{UNID}{FIM}                    # 500ML, 1,5 L, CX10KG
  | \b{EMB}\s?[/-]\s?\d+(?:\s?[/-]\s?{EMB})?{FIM}           # CX/12, UN/0001, CX-6-UN
  | \b{EMB}\d+{EMB}?{FIM}                                   # PT0001UN
  | \bC/\s?\d+(?:\s?{UNID})?{FIM}                           # C/12, C/20MTS
  | \bQTD\.?\s?{NUM}(?:\s{EMB})?                            # QTD. 2.00 DP
  | \(\s?\d+\s?{UNID}?\s?\)                                 # (12 UN)
  | \(CX[\d,]+(?:M2|PCS?)\)                                 # (CX1,62M2)
  | -\s?\d+VL\b                                             # - 24VL
  | \bF\.?P\.?S\.?\s?-?\s?\d+\+?                            # FPS 70
  | \bLV?\d+\s?PG?\d+(?:ML?|G|GR)?\b                        # LV12PG11, L4P3, LV16 PG15
  | \b\d+\s?{EMB}\s?\d+\s?{UNID}{FIM}                       # 1CX12UN
''', re.X)
ALFANUM = re.compile(r'(?=.*\d)(?=.*[a-z])[a-z\d]+')   # tem letra e dígito
FRAG    = re.compile(r'[a-z]{3,}')                      # fragmento alfabético resgatável

# '.' e '/' usados como separador (não quebra decimais e frações: 2.5mm, 1/2)
SEP = re.compile(r'(?<!\d)[./]|[./](?!\d)')
TABELA_PONT = str.maketrans('', '', string.punctuation)
# número+unidade dentro de token (CAB.SAV.750ML -> 750ml, 1-KG -> 1kg, 12x40g)
QTD_TOKEN = re.compile(r'(?:\d+x)*\d+(?:m2|mln|ml|mls|m|mt|mts|g|gr|grs|kg|kgs|l|lt|lts|li|un|und|unid|pc|pcs)')
# palavra colada a uma quantidade (FUGINI100GR -> fugini + 100gr; HARIBO12X350G -> haribo + 12x350g)
COLADO = re.compile(r'([a-z]{2,})((?:\d+x)*\d+(?:m2|mln|ml|mls|m|mt|mts|g|gr|grs|kg|kgs|l|lt|lts|li|un|und|unid|pc|pcs))')

# tokens só de dígitos, separadores e 'x': 3, 1/2, 030003, 20x25
NUMERICO = re.compile(r'[\d.,/+\-%*:;"\'&#x]*\d[\d.,/+\-%*:;"\'&#x]*')
# códigos de referência (preserva DN15, RJ11, IP65, E27, MM2, CA50 e medidas como 2700K, 6800W)
CODIGO = re.compile(r'(?:ref|cod|mva|lote|venc|nr|n\.)[.:\-]?\w[\w.\-/]*'          # REF 123, MVA:29
                    r'|[a-z]-?\d+[a-z]?'                                          # C21, R0091, S08
                    r'|[a-z]-\d+/\d+'                                             # B-4472/17
                    r'|\d{4,}-?(?!(?:k|w|v|a|kv|ka|va|mm|cm|m)$)[a-z]{1,2}'        # 27814E, 1328-M
                    r'|[a-z]{1,2}\d{4,}'                                          # MK2431
                    r'|r\.\d+|a/\d+|tam\d+|\d+h\d+'                               # R.123, A/4552, TAM2, 1H15
                    r'|(?!dn|rj|ip|mm|ca|e\d)[a-z]{2}-?\d+[a-z]?')                # SG-8407, GL83, BR-01
# dimensões NxM(xP), com ou sem unidade: 20x25, 67x370mm, 32mmx22mm, 2750x1840x15mm, 19mmx10m
DIMENSAO = re.compile(r'(?:\d+(?:mm|cm|mts?|m)?x)+\d+(?:mm|cm|mts?|m)?')

def _sem_acento(s): return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()

RUIDO = (
    {'ml','mls','gr','grs','kg','kgs','lt','lts','un','und','unid','cx','cxa','fd','dp','pct','pc','pcs',
     'mt','mts','pt','fr','sc','bd','gf'}                                         # unidades e embalagens
  | {'pack','sache','refil','cartela','pet','lata'}                               # embalagem
  | {'premium','prem','especial','classic','classico','class','super','ultra','plus','max','maxi','top',
     'gold','golden','star','prime','master','power','smart','easy','life','ideal','royal','imperial',
     'new','novo','nova','tradic','orig','duo','one'}                             # marketing / linha
  | {'grande','gde','pequeno','pequena','peq'}                                    # tamanho
  | {'ref','cod','nr'}                                                            # referência
  | ({_sem_acento(w) for w in stopwords.words('portuguese')} - {'com','sem','nao'})
)

@lru_cache(maxsize=None)
def _tokeniza(df: pd.Series):
    # 1. Remover acentos e quantidades/tamanhos de embalagem (sobre a string)
    texto = PADRAO_QTD.sub(' ', _sem_acento(df).upper())

    # 2. Dividir o texto em tokens
    tokens = word_tokenize(texto)

    # 3. Converter o texto para letras minúsculas
    tokens = [w.lower() for w in tokens]

    # 4. Separar tokens que usam '.' ou '/' como separador (VIN.PORT.CHAMINE, P/TRAT)
    tokens = [p for w in tokens for p in SEP.split(w)]

    # 5. Remover símbolos de pontuação de cada token e descartar tokens vazios
    tokens = [w.translate(TABELA_PONT) for w in tokens]
    tokens = [w for w in tokens if w]

    # 6. Separar palavra colada a uma quantidade (fugini100gr -> fugini, 100gr)
    tokens = [p for w in tokens for m in [COLADO.fullmatch(w)] for p in (m.groups() if m else (w,))]

    # 7. Remover unidades, embalagens, marketing, tamanho, referência e stop words
    tokens = [w for w in tokens if w not in RUIDO]

    # 8. Remover letras soltas
    tokens = [w for w in tokens if not (len(w) == 1 and w.isalpha())]

    # 9. Remover números soltos (3, 1/2, 030003), quantidades (750ml, 1kg) e dimensões (67x370mm)
    tokens = [w for w in tokens if not NUMERICO.fullmatch(w)
                               and not QTD_TOKEN.fullmatch(w)
                               and not DIMENSAO.fullmatch(w)]

    # 10. Remover códigos de referência (REF 123, C21, SG-8407, 27814E)
    tokens = [w for w in tokens if not CODIGO.fullmatch(w)]

    return tokens
  
# Em quantas descrições da base cada token aparece (calculado uma vez)
DF_TOKENS = Counter(t for toks in base['XPROD'].map(_tokeniza) for t in set(toks))

def resgata(tokens):
    """Alfanuméricos que aparecem em no máximo uma descrição da base são trocados
    pelos seus fragmentos alfabéticos (>= 3 letras, fora do RUIDO); sem fragmento, somem."""
    out = []
    for w in tokens:
        if DF_TOKENS[w] <= 1 and ALFANUM.fullmatch(w):
            out += [p for p in FRAG.findall(w) if p not in RUIDO]
        else:
            out.append(w)
    return out

def preprocessamento(df: pd.Series):
    # Tokeniza e resgata fragmentos de alfanuméricos únicos
    return resgata(_tokeniza(df))

[nltk_data] Downloading package punkt_tab to /home/Leo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/Leo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
#@title Calcula a matriz TF-IDF do dataset completo
import os, io, urllib.request
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer


REPO_RAW  = 'https://raw.githubusercontent.com/LeoKiepper/Ciencia-de-Dados/main/trabalho2/'
ARQ_X     = 'X_tfidf.npz'
ARQ_VOCAB = 'X_tfidf_vocab.csv'

def calculaX(x: pd.Series):
    """TF-IDF de x. Devolve (M, vocab): M é CSR float32 (linhas na ordem de x), vocab é o array de tokens."""
    vectorizer = TfidfVectorizer(analyzer=preprocessamento, dtype=np.float32)
    M = vectorizer.fit_transform(x)
    return M, vectorizer.get_feature_names_out()

def carregaX(x: pd.Series):
    """Carrega (M, vocab): arquivo local -> GitHub -> calcula com calculaX e salva."""
    if os.path.exists(ARQ_X) and os.path.exists(ARQ_VOCAB):
        return sp.load_npz(ARQ_X), pd.read_csv(ARQ_VOCAB, keep_default_na=False)['token'].to_numpy()
    try:
        M = sp.load_npz(io.BytesIO(urllib.request.urlopen(REPO_RAW + ARQ_X).read()))
        return M, pd.read_csv(REPO_RAW + ARQ_VOCAB, keep_default_na=False)['token'].to_numpy()
    except Exception:
        pass
    M, vocab = calculaX(x)
    sp.save_npz(ARQ_X, M)
    pd.Series(vocab, name='token').to_csv(ARQ_VOCAB, index=False)
    print(f'Matriz calculada e salva em {ARQ_X} e {ARQ_VOCAB}')
    return M, vocab




In [5]:
#@title Define variáveis de entrada para os algoritimo de classificação
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

X_full, vocab = carregaX(base['XPROD'])         # matriz do dataset completo (cache)

USAR_AMOSTRA = False
N_AMOSTRA = 1000
TEST_SIZE = 0.25
SEMENTE_AMOSTRA = None          # None: sorteia; inteiro (ex.: 2003343720): fixa
SEMENTE_SPLIT = None            # None: sorteia; inteiro: fixa

if USAR_AMOSTRA:
    semente_amostra = SEMENTE_AMOSTRA if SEMENTE_AMOSTRA is not None else int(np.random.randint(0, 2**31))
    df = base.sample(n=N_AMOSTRA, random_state=semente_amostra)
else:
    semente_amostra = None
    df = base

X = X_full[base.index.get_indexer(df.index)]    # linhas de X_full na ordem de df
le = LabelEncoder()
y = pd.Series(le.fit_transform(df['NCM']), index=df.index, name='NCM')

print(f'X {X.shape} {X.dtype}, {X.nnz} valores não nulos')
show(comoDataFrame(X, vocab, indice=df.index))
print(pd.Series(vocab).sample(50).tolist())


# Separa uma fatia do dataset para teste
semente_split = SEMENTE_SPLIT if SEMENTE_SPLIT is not None else int(np.random.randint(0, 2**31))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=semente_split)

for nome, obj in {'X_train': X_train, 'X_test': X_test,
                  'y_train': y_train, 'y_test': y_test, 'le': le}.items():
    salva(nome, obj)

# Com alguma semente sorteada, os arquivos do split são temporários: a avaliação os remove após gravar o log
salva('config_dados', {'usar_amostra': USAR_AMOSTRA,
                       'n_amostra': len(df),
                       'semente_amostra': semente_amostra,
                       'test_size': TEST_SIZE,
                       'semente_split': semente_split,
                       'guardou_matrizes': not ((USAR_AMOSTRA and SEMENTE_AMOSTRA is None) or SEMENTE_SPLIT is None)})


X (1117623, 84587) float32, 4937893 valores não nulos


['solta', 'carbonar', 'cobrekabos', '192mm', 'duch', 'cebebola', 'mokaccino', 'reservad', 'r6323zlo', 'espesso', 'sawhorse', 'piko', 'finisherfresh', 'arjasmim', 'rechado', 'naciona', 'perfuradou', '150n', 'adstring', 'damper', '584x117a', 'sosho', 'plasutilxicara', 'latd', 'labadi', 'homerum', 'fata', 'vq', 'tforce', 'enrolado', 'inoxpinca', 'sna', 'chocbranco', 'ssuporte', '32hda04', 'underberg', 'fenzza', 'harris', 'annita', 'geoffroy', 'balmer', 'gsb510', 'r63981ni', 'yogurmix', 'souly', '45x200r001', 'vuole', 'pernambuco', 'johnny', 'buteco']


In [ ]:
#@title Suite de treino: validação cruzada -> melhores parâmetros -> modelo retreinado
import time
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV

def _treina(estimador, param_grid, X_train, y_train, nome=None,
           cv=5, scoring=None, n_jobs=-1, verbose=1, refaz=False):
    """GridSearchCV sobre param_grid; com refit=True, retreina o melhor em todo X_train.
    Salva modelo_<nome> (só o modelo) e treino_<nome> (metadados). Recebe apenas dados de treino.
    Se já existem e refaz=False, não treina de novo. Devolve o nome."""
    nome = nome or type(estimador).__name__
    if existe(f'modelo_{nome}') and existe(f'treino_{nome}') and not refaz:
        print(f'[{nome}] já treinado em {DIR_ART}/')
        return nome

    busca = GridSearchCV(clone(estimador), param_grid, cv=cv, scoring=scoring,
                         n_jobs=n_jobs, refit=True, verbose=verbose)
    t0 = time.perf_counter()
    busca.fit(X_train, y_train)
    tempo = time.perf_counter() - t0
    salva(f'modelo_{nome}', busca.best_estimator_)
    salva(f'treino_{nome}', {
        'melhores_parametros': busca.best_params_,
        'parametros': busca.best_estimator_.get_params(),
        'melhor_score_cv': busca.best_score_,
        'cv_results': pd.DataFrame(busca.cv_results_),      # vetores da validação cruzada, para plots
        'tempo_s': tempo,
        'indice_treino': y_train.index,                     # linhas da base usadas no treino (auditoria)
    })
    print(f'[{nome}] {tempo:.0f} s; melhores: {busca.best_params_}')
    return nome

def treina(pares, X_train, y_train, **opcoes):
    """pares: lista de (estimador, param_grid). Devolve a lista de nomes treinados."""
    return [_treina(estimador, param_grid, X_train, y_train, **opcoes) for estimador, param_grid in pares]


In [ ]:
#@title Treina os algoritmos
import numpy as np
from sklearn.ensemble import RandomForestClassifier

X_train, y_train = carrega('X_train'), carrega('y_train')


RF_PRODUTO = 150_000
RF_L = np.geomspace(start=1875, stop=60_000, num=19).round(-1).astype(int)
RF_pares = sorted({(max(1, round(RF_PRODUTO / L)), int(L)) for L in RF_L})
RF_base = {
    'max_features': [64, 'sqrt'],
    'class_weight': [None, 'balanced_subsample'],
    'bootstrap': [True, False]}



MODELOS = [
    (RandomForestClassifier(n_jobs=20),
     [{**RF_base, 'n_estimators': [T], 'max_leaf_nodes': [L]} for T, L in RF_pares]),
]

nomes = treina(MODELOS, X_train, y_train,
               scoring='f1_macro', n_jobs=1, refaz=True, verbose=3)


Fitting 5 folds for each of 72 candidates, totalling 360 fits


/home/Leo/.conda/envs/python_dev-general/lib/python3.13/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


[RandomForestClassifier] 238 s; melhores: {'max_depth': 8, 'max_features': 1.0, 'max_samples': 0.6, 'n_estimators': 1000}


In [ ]:
#@title Inferência: ŷ_* = A(X_*)
import os, glob, time, joblib

def infere(modelo, X):
    """Dado um modelo treinado e um X, devolve ŷ."""
    return modelo.predict(X)


SUFIXOS = ['test']
for arq in sorted(glob.glob(_caminho('modelo_*', 'joblib'))):
    nome = os.path.basename(arq)[len('modelo_'):-len('.joblib')]
    modelo = joblib.load(arq)
    for suf in SUFIXOS:     # O ŷ herda o sufixo que o seu respectivo X tinha
        t0 = time.perf_counter()
        salva(f'yhat_{suf}_{nome}', infere(modelo, carrega(f'X_{suf}')))
        print(f'[{nome}] yhat_{suf}: {time.perf_counter() - t0:.1f} s')


[RandomForestClassifier] yhat_test: 0.1 s


In [100]:
#@title Função de avaliação
from sklearn.metrics import classification_report

def avalia(y_true, y_pred):
    """Compara y_true com ŷ. Devolve {medida: resultado}; novas medidas entram como novas chaves."""
    return {
        'classification_report': classification_report(y_true, y_pred, output_dict=True, zero_division=0),
    }


In [102]:
#@title Avalia: y_test vs ŷ_test, impresso e registrado em artefatos/avaliacao_<timestamp>.log
import os, sys, glob, datetime, hashlib, joblib
import numpy as np, scipy, sklearn
import pandas as pd

agora = datetime.datetime.now()
ARQ_LOG = _caminho(f'avaliacao_{agora:%Y%m%d_%H%M%S}', 'log')

config = carrega('config_dados')
with open(ARQ_X, 'rb') as f:                    # identifica a matriz TF-IDF usada
    sha_x = hashlib.sha256(f.read()).hexdigest()

y_test = carrega('y_test')
linhas = [f'=== Avaliação {agora:%Y-%m-%d %H:%M:%S} ===',
          f'Dados: {config}',
          f'Matriz TF-IDF: {ARQ_X} sha256={sha_x}',
          f'Versões: python {sys.version.split()[0]}, numpy {np.__version__}, '
          f'scipy {scipy.__version__}, scikit-learn {sklearn.__version__}', '']

for arq in sorted(glob.glob(_caminho('treino_*', 'joblib'))):
    nome = os.path.basename(arq)[len('treino_'):-len('.joblib')]
    treino = joblib.load(arq)

    # Auditoria: nenhuma linha de teste pode ter sido usada no treino
    comuns = treino['indice_treino'].intersection(y_test.index)
    assert len(comuns) == 0, f'[{nome}] {len(comuns)} linhas de teste foram usadas no treino'

    resultado = avalia(y_test, carrega(f'yhat_test_{nome}'))
    linhas += [f'--- {nome} ---',
               f'Melhores parâmetros (CV): {treino["melhores_parametros"]}',
               f'Score médio na CV: {treino["melhor_score_cv"]:.4f}; tempo de treino: {treino["tempo_s"]:.0f} s',
               f'Parâmetros completos: {treino["parametros"]}',
               pd.DataFrame(resultado['classification_report']).T.to_string(), '']

texto = '\n'.join(linhas)
print(texto)
with open(ARQ_LOG, 'w') as f:
    f.write(texto + '\n')

# Semente sorteada: o split só existia para manter o fluxo de arquivos; as sementes estão no log
if not config['guardou_matrizes']:
    for nome in ('X_train', 'X_test', 'y_train', 'y_test', 'config_dados'):
        for ext in ('npz', 'joblib'):
            if os.path.exists(_caminho(nome, ext)):
                os.remove(_caminho(nome, ext))
    print(f'Arquivos temporários do split removidos; sementes registradas em {ARQ_LOG}')


=== Avaliação 2026-11-15 21:14:34 ===
Dados: {'usar_amostra': True, 'n_amostra': 1000, 'semente_amostra': 239575791, 'test_size': 0.25, 'semente_split': 1438220465, 'guardou_matrizes': False}
Matriz TF-IDF: X_tfidf.npz sha256=bef63211aea005880f69c96490f63c20cd61fd694f1b10723976c09453b24969
Versões: python 3.13.13, numpy 2.4.6, scipy 1.18.0, scikit-learn 1.9.1

--- RandomForestClassifier ---
Melhores parâmetros (CV): {'max_depth': 8, 'max_features': 1.0, 'max_samples': 0.6, 'n_estimators': 1000}
Score médio na CV: 0.2520; tempo de treino: 238 s
Parâmetros completos: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': 0.6, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 1000, 'n_jobs': None, 'oob_score': False, 'random_state': None, 'verbose': 0, 'warm_start': False}
              prec